# Grokipedia vs Wikipedia — scaricamento, confronto e verifica di paternità

Scarica i 40 articoli di Grokipedia corrispondenti a quelli già usati per Wikipedia in
`altmann_wikipedia.ipynb`, li ripulisce, li confronta e verifica che **la versione attuale**
sia scritta da Grok e non adattata da Wikipedia.

## Perché non si usa il dump

Il dump `htriedman/grokipedia-v0.1-dump` è inservibile per questo scopo, per due ragioni
indipendenti, entrambe misurate:

1. **Il testo è rovinato.** Dove c'erano dei link le parole sono incollate:
   `"TheSimca 1301andSimca 1501are two relatedautomobileswhich were produced by
   theFrenchautomakerSimcafrom 1966"`. L'indicatore *camelCase per token* vale 0.0165 contro
   0.0004 di Wikipedia, ed è una stima per difetto perché gli incollamenti fra due minuscole
   (`relatedautomobileswhich`) non lasciano traccia di maiuscola. È lo stesso modo di
   fallimento del troncamento sugli accenti: il contatore di frequenze e il cercatore di
   posizioni non vedono le stesse parole, e le occorrenze perse si concentrano sui nomi
   propri, cioè sulle keyword.
2. **Mancano gli articoli che servono.** Dei 40 titoli qui usati il dump ne contiene **uno**
   (`Rome`). `title LIKE 'Napoleon%'` restituisce solo `Napoleon_Abueva`; `Italy%` e
   `Slavery%` zero. Il dump contiene inoltre solo Grokipedia (`source='grok'` 4.948.358
   chunk, `source='wikipedia'` 0).

Si scarica quindi dal sito, dove il testo è pulito e tutti e 40 gli articoli esistono.

## Come viene estratto il testo

Il corpo dell'articolo è pre-renderizzato lato server dentro `<span data-tts-block="true">`
(il commento nel sorgente dice *"Article body — pre-rendered HTML from Rust, ZERO client JS"*).
Prendendo quei blocchi si evitano tre contaminazioni che l'estrazione ingenua del DOM porta
con sé, tutte verificate:

- **l'indice di navigazione**, che occupa i primi ~2000 caratteri ed è la concatenazione di
  tutti i titoli di sezione: un ammasso dei termini chiave dell'articolo posizionato esattamente
  all'inizio della finestra di troncamento, cioè un burst artificiale dove più farebbe danno;
- **il footer** con il modulo di segnalazione;
- **i marcatori di citazione `[n]`**, 772 in *Napoleon*, 269 in *Slavery*, 411 in *Moon*.
  L'estratto dell'API di Wikipedia non li contiene, e siccome qui il tempo si conta in
  caratteri lasciarli renderebbe i due flussi non confrontabili.

## Sulle revisioni

**Grokipedia non pubblica una cronologia delle revisioni.** Gli unici endpoint esposti dal
sito sono `/api/analytics`, `/api/create-article-request`, `/api/create-edit-request`,
`/api/ev`, `/api/tts`: nessun `history`, `versions` o `revisions` (tutti 404). Nella pagina
compaiono solo `dateModified` nel JSON-LD e la dicitura *"Fact-checked by Grok N ago"*.

Si riportano quindi tre cose distinte, senza spacciarle per un conteggio di edit:

1. `dateModified` dichiarato da Grokipedia;
2. il numero di **catture** su Internet Archive e il loro intervallo — sono catture, non
   revisioni: il digest dell'HTML cambia a ogni visita per via degli elementi dinamici, quindi
   contare i digest distinti darebbe un numero privo di significato;
3. **quanto il testo è effettivamente cambiato** fra la prima e l'ultima cattura archiviata,
   misurato sul testo estratto con lo stesso estrattore. Questa è l'unica misura di revisione
   difendibile che si possa ottenere dall'esterno.

## 1. Configurazione

In [ ]:
from pathlib import Path

QUI = Path.cwd()
CACHE_WIKI = QUI / "cache_wikipedia"        # popolata da altmann_wikipedia.ipynb
CACHE_GROK = QUI / "cache_grokipedia"
CACHE_WB   = QUI / "cache_wayback"
OUT_DIR    = QUI / "risultati_confronto"

N_EFF = 80_000          # tutte le 40 coppie superano questa soglia da entrambi i lati
SHINGLE_N = 8           # lunghezza degli n-grammi di parole per la sovrapposizione
SOGLIA_COPIA = 0.20     # oltre questa frazione l'articolo si considera derivato da Wikipedia

FAI_WAYBACK = True      # confronto prima/ultima cattura su Internet Archive (lento)
WB_TIMEOUT  = 120

PAUSA_GROK = 0.8        # secondi fra richieste a grokipedia.com
UA_BROWSER = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/122.0 Safari/537.36")
UA_RICERCA = "AltmannBurstinessResearch/1.0 (tesi universitaria) python-urllib"

# i 40 titoli sono quelli selezionati da altmann_wikipedia.ipynb
TITOLI_CSV = QUI / "risultati_wiki" / "dati" / "articoli_qualita.csv"
N_ARTICOLI = 40
print("cartella:", QUI.resolve())

In [ ]:
import re, ssl, json, time, gzip, zlib, hashlib, html as htmlmod
import urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

try:
    import certifi; HAS_CERTIFI = True
except Exception:
    HAS_CERTIFI = False

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": .25})

for d in (CACHE_GROK, CACHE_WB, OUT_DIR, OUT_DIR / "dati", OUT_DIR / "figure"):
    d.mkdir(parents=True, exist_ok=True)

def contesto_ssl():
    if HAS_CERTIFI:
        try: return ssl.create_default_context(cafile=certifi.where())
        except Exception: pass
    try: return ssl.create_default_context()
    except ssl.SSLError:
        c = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        c.check_hostname = False; c.verify_mode = ssl.CERT_NONE
        print("ATTENZIONE: verifica del certificato disattivata; installare certifi")
        return c
CTX = contesto_ssl()

WORD_CHAR = r"[^\W\d_]"
TOKEN_RE  = re.compile(WORD_CHAR + r"+(?:['\u2019\-]" + WORD_CHAR + r"+)*", re.UNICODE)
CAMEL     = re.compile(r"[a-z][A-Z]")
FRASE     = re.compile(r"[.!?][\"'\u2019\)\]]*(?:\s|$)")
PAROLA    = re.compile(r"[a-z']+")

Q = pd.read_csv(TITOLI_CSV)
# I 40 titoli dell'analisi sono quelli di riepilogo.csv (13 agosto 2026). La
# selezione per lunghezza da sola non e' stabile: le tre voci di Wikipedia
# scaricate il 15 agosto hanno cambiato articoli_qualita.csv, e con quel file
# 'Istanbul' entrerebbe fra le 40 al posto di 'Rome'.
FISSATI = OUT_DIR / "dati" / "riepilogo.csv"
if FISSATI.exists():
    TITOLI = pd.read_csv(FISSATI)["titolo"].tolist()
else:
    TITOLI = (Q[Q["ammesso"]].sort_values("n_char", ascending=False)
              .head(N_ARTICOLI)["titolo"].tolist())
LEN_WIKI = dict(Q.set_index("titolo")["n_char"])
print(f"{len(TITOLI)} titoli, dal piu' lungo: {TITOLI[:4]} ...")

## 2. Estrattore

`data-tts-block` isola il corpo dell'articolo. I marcatori `[n]` vengono rimossi per simmetria
con l'estratto di Wikipedia, che non li contiene.

In [ ]:
TTS = re.compile(r'<span[^>]*data-tts-block="true"[^>]*>(.*?)</span>', re.S)
LD  = re.compile(r'<script[^>]*type="application/ld\+json"[^>]*>(.*?)</script>', re.S)

def _frag(h):
    h = re.sub(r"(?is)<(script|style|button|svg)[^>]*>.*?</\1>", " ", h)
    h = re.sub(r"<[^>]+>", " ", h)
    h = htmlmod.unescape(h)
    h = re.sub(r"\[\d+\]", " ", h)             # marcatori di citazione
    h = re.sub(r"[ \t\u00a0]+", " ", h)
    return h.strip()

ARTICOLO = re.compile(r"<article[^>]*>(.*?)</article>", re.S)

def estrai_grok(page):
    """Corpo dell'articolo, con due strategie in cascata.

    Il sito ATTUALE pre-renderizza il corpo dentro <span data-tts-block="true">.
    Le pagine ARCHIVIATE su Internet Archive (ottobre 2025, v0.1) non hanno quei
    marcatori: li' il corpo sta dentro <article>. Senza il secondo tentativo
    l'estrazione degli snapshot restituisce stringa vuota, e il confronto storico
    non produce nulla senza dirlo."""
    b = [_frag(x) for x in TTS.findall(page)]
    b = [x for x in b if len(x) > 40]
    if b:
        return re.sub(r"\n{3,}", "\n\n", "\n\n".join(b)).strip(), "tts"
    m = ARTICOLO.search(page)
    if m:
        return _frag(m.group(1)), "article"
    return "", "nessuno"

def testo_grok(page):
    return estrai_grok(page)[0]

def decodifica(b):
    """Gli snapshot 'id_' di Wayback restituiscono i byte ORIGINALI della cattura,
    che sono quasi sempre gzip: urllib non li decomprime da solo. Senza questo
    controllo si ottiene un blob binario che, passato allo strip dei tag, produce
    un finto testo con lunghezza media dei token ~1.3 caratteri."""
    if b[:2] == b"\x1f\x8b":
        for f in (gzip.decompress, lambda x: zlib.decompress(x, 16 + zlib.MAX_WBITS)):
            try:
                b = f(b); break
            except Exception:
                pass
    return b.decode("utf-8", errors="replace")

def prosa_valida(t, min_char=20000):
    """gate: distingue prosa vera da estrazioni degenerate"""
    tk = TOKEN_RE.findall(t)
    if len(t) < min_char or len(tk) < 200:
        return False
    lt = float(np.mean([len(w) for w in tk]))
    fr = 1000 * len(FRASE.findall(t)) / len(t)
    return 4.0 <= lt <= 8.0 and fr >= 2.0

def meta_grok(page):
    """dateModified dal JSON-LD e la dicitura 'Fact-checked by Grok ...'"""
    out = {}
    for m in LD.finditer(page):
        try: d = json.loads(m.group(1))
        except Exception: continue
        if isinstance(d, dict) and d.get("@type") == "Article":
            out["dateModified"] = d.get("dateModified")
            out["headline"] = d.get("headline")
    testo = re.sub(r"<[^>]+>", " ", page[:200_000])
    testo = re.sub(r"\s+", " ", htmlmod.unescape(testo))
    m = re.search(r"Fact-checked by Grok\s+([^A-Z]{2,40}?)\s+[A-Z]", testo)
    out["fact_checked"] = m.group(1).strip() if m else None
    return out

def indicatori(t):
    tk = TOKEN_RE.findall(t); low = [w.lower() for w in tk]
    if not tk: return dict(n_char=len(t), camel=np.nan, frasi_1k=np.nan, len_token=np.nan)
    return dict(n_char=len(t),
                camel=len(CAMEL.findall(t)) / len(tk),
                frasi_1k=1000 * len(FRASE.findall(t)) / len(t),
                len_token=float(np.mean([len(w) for w in low])))

def shingles(t, n=SHINGLE_N):
    w = PAROLA.findall(t.lower())
    return {" ".join(w[i:i+n]) for i in range(max(len(w) - n + 1, 0))}

print("estrattore pronto")

## 3. Scaricamento da grokipedia.com

In [ ]:
def scarica_grok(titolo):
    slug = titolo.replace(" ", "_")
    fn = CACHE_GROK / (re.sub(r"[^A-Za-z0-9_]", "", slug)[:60] + ".html")
    if fn.exists():
        return fn.read_bytes().decode("utf-8", errors="replace"), True
    url = "https://grokipedia.com/page/" + urllib.parse.quote(slug)
    req = urllib.request.Request(url, headers={"User-Agent": UA_BROWSER})
    with urllib.request.urlopen(req, timeout=90, context=CTX) as r:
        page = r.read().decode("utf-8", errors="replace")
    fn.write_bytes(page.encode("utf-8"))      # binario: niente traduzione dei newline
    time.sleep(PAUSA_GROK)
    return page, False

GROK, META = {}, {}
nuovi = 0
print(f"scarico {len(TITOLI)} articoli...")
for i, t in enumerate(TITOLI, 1):
    try:
        page, da_cache = scarica_grok(t)
        nuovi += (not da_cache)
        GROK[t] = testo_grok(page)
        META[t] = meta_grok(page)
    except Exception as e:
        print(f"\n  errore su '{t}': {type(e).__name__}: {e}")
    if i % 10 == 0: print(f"  {i}/{len(TITOLI)}", end="", flush=True)
print(f"\narticoli ottenuti: {len(GROK)}/{len(TITOLI)}  (scaricati ora: {nuovi})")
_corti = [t for t, v in GROK.items() if len(v) < N_EFF]
if _corti:
    print(f"ATTENZIONE: sotto {N_EFF:,} caratteri: {_corti}")

## 4. Wikipedia, dalla cache del notebook precedente

In [ ]:
APP = re.compile(r"^==+\s*(See also|References|Notes|Citations|Sources|Bibliography|"
                 r"Further reading|External links|Works cited|Footnotes|Explanatory notes|"
                 r"General sources)\s*==+\s*$", re.M | re.I)
INT = re.compile(r"^==+.*?==+\s*$", re.M)

def carica_wiki(t):
    fn = CACHE_WIKI / (hashlib.sha256(t.encode()).hexdigest()[:14] + ".txt")
    if not fn.exists():
        return None
    tx = fn.read_bytes().decode("utf-8", errors="replace")
    m = APP.search(tx)
    if m: tx = tx[:m.start()]
    tx = INT.sub("", tx)
    tx = re.sub(r"\n[ \t]+\n", "\n\n", tx)
    return re.sub(r"\n{3,}", "\n\n", tx).strip()

WIKI = {t: carica_wiki(t) for t in TITOLI}
mancanti = [t for t, v in WIKI.items() if not v]
if mancanti:
    print(f"mancano dalla cache ({len(mancanti)}): {mancanti[:6]}")
    print("-> eseguire prima altmann_wikipedia.ipynb")
COPPIE = [t for t in TITOLI if WIKI.get(t) and GROK.get(t)]
print(f"coppie complete: {len(COPPIE)}/{len(TITOLI)}")

## 5. Confronto del testo

Gli indicatori sono calcolati sulla finestra comune di `N_EFF` caratteri, la stessa che userà
l'analisi di Altmann. La *copertura di vocabolario* di ciascun articolo è misurata contro il
vocabolario del **corrispondente articolo di Wikipedia**: è un confronto appaiato per tema, e
quindi non risente della deriva tematica come farebbe un vocabolario di riferimento unico.

In [ ]:
righe = []
for t in COPPIE:
    w, g = WIKI[t][:N_EFF], GROK[t][:N_EFF]
    vw = set(x.lower() for x in TOKEN_RE.findall(WIKI[t]))
    vg = set(x.lower() for x in TOKEN_RE.findall(GROK[t]))
    tw = [x.lower() for x in TOKEN_RE.findall(w)]
    tg = [x.lower() for x in TOKEN_RE.findall(g)]
    r = dict(titolo=t,
             wiki_tot=len(WIKI[t]), grok_tot=len(GROK[t]),
             rapporto=len(GROK[t]) / len(WIKI[t]))
    for nome, txt, tok, altro_voc in [("wiki", w, tw, vg), ("grok", g, tg, vw)]:
        ind = indicatori(txt)
        r[f"{nome}_camel"] = ind["camel"]
        r[f"{nome}_frasi1k"] = ind["frasi_1k"]
        r[f"{nome}_lentok"] = ind["len_token"]
        r[f"{nome}_voc_altro"] = sum(x in altro_voc for x in tok) / max(len(tok), 1)
    righe.append(r)
C = pd.DataFrame(righe)
C.to_csv(OUT_DIR / "dati" / "confronto_testo.csv", index=False)

print(f"Confronto su finestra comune di {N_EFF:,} caratteri ({len(C)} coppie)\n")
def rs(col_w, col_g, nome, fmt="{:.4f}"):
    a, b = C[col_w], C[col_g]
    print(f"  {nome:34s} wiki " + fmt.format(a.median()) +
          f"  [{fmt.format(a.min())}, {fmt.format(a.max())}]   "
          f"grok " + fmt.format(b.median()) +
          f"  [{fmt.format(b.min())}, {fmt.format(b.max())}]")
rs("wiki_camel", "grok_camel", "camelCase per token")
rs("wiki_frasi1k", "grok_frasi1k", "frasi per 1000 caratteri", "{:.2f}")
rs("wiki_lentok", "grok_lentok", "lunghezza media dei token", "{:.2f}")
rs("wiki_voc_altro", "grok_voc_altro", "copertura sul vocabolario dell'altro", "{:.3f}")
print(f"\n  lunghezza totale: wiki mediana {C['wiki_tot'].median():,.0f} | "
      f"grok mediana {C['grok_tot'].median():,.0f} | "
      f"rapporto mediano {C['rapporto'].median():.2f} "
      f"[{C['rapporto'].min():.2f}, {C['rapporto'].max():.2f}]")
print(f"  coppie con entrambi >= {N_EFF:,}: "
      f"{int(((C['wiki_tot'] >= N_EFF) & (C['grok_tot'] >= N_EFF)).sum())}/{len(C)}")

## 6. La versione attuale è scritta da Grok?

Sovrapposizione di $n$-grammi di parole ($n = 8$) fra i due testi completi. Un $n$-gramma di 8
parole consecutive identiche non nasce per caso su testi indipendenti: è la misura standard per
la quasi-duplicazione. Si riporta la frazione degli $n$-grammi di Grokipedia presenti anche in
Wikipedia — la direzione che conta per la domanda *"Grok ha copiato?"*.

In [ ]:
righe = []
for t in COPPIE:
    sw, sg = shingles(WIKI[t]), shingles(GROK[t])
    if not sw or not sg: continue
    inter = len(sw & sg)
    righe.append(dict(titolo=t, n_wiki=len(sw), n_grok=len(sg),
                      grok_in_wiki=inter / len(sg),
                      wiki_in_grok=inter / len(sw),
                      jaccard=inter / len(sw | sg)))
S = pd.DataFrame(righe).sort_values("grok_in_wiki", ascending=False)
S.to_csv(OUT_DIR / "dati" / "sovrapposizione.csv", index=False)

print(f"Sovrapposizione di {SHINGLE_N}-grammi (frazione di Grokipedia presente in Wikipedia)\n")
for r in S.head(8).itertuples():
    print(f"  {r.titolo:26s} {r.grok_in_wiki:.4f}")
print(f"  ... (le altre {len(S)-8} sotto {S['grok_in_wiki'].iloc[7]:.4f})")
print(f"\n  mediana {S['grok_in_wiki'].median():.4f} | massimo {S['grok_in_wiki'].max():.4f} "
      f"({S.iloc[0]['titolo']})")
for s in (0.02, 0.05, 0.10, SOGLIA_COPIA):
    print(f"  sopra {s:.0%}: {int((S['grok_in_wiki'] > s).sum())}/{len(S)}")
DERIVATI = S[S["grok_in_wiki"] > SOGLIA_COPIA]["titolo"].tolist()
print(f"\nVERDETTO: {len(S) - len(DERIVATI)}/{len(S)} articoli sono scritti da Grok "
      f"(sovrapposizione <= {SOGLIA_COPIA:.0%}).")
if DERIVATI:
    print(f"  da escludere perche' derivati da Wikipedia: {DERIVATI}")

## 7. Revisioni

In [ ]:
print("Grokipedia non espone una cronologia delle revisioni: gli unici endpoint del sito")
print("sono /api/analytics, /api/create-article-request, /api/create-edit-request, /api/ev,")
print("/api/tts. Le richieste a /history, /versions, /revisions restituiscono 404.\n")
print("Si riporta quanto e' effettivamente osservabile.\n")

M = pd.DataFrame([dict(titolo=t, **META.get(t, {})) for t in COPPIE])
M["dateModified"] = pd.to_datetime(M["dateModified"], errors="coerce", utc=True)
M = M.sort_values("dateModified")
print("dateModified dichiarato da Grokipedia:")
print(f"  intervallo: {M['dateModified'].min()}  ->  {M['dateModified'].max()}")
print(f"  mediana:    {M['dateModified'].median()}")
_fc = M["fact_checked"].dropna()
if len(_fc):
    print(f"  dicitura 'Fact-checked by Grok': {_fc.value_counts().head(6).to_dict()}")
M.to_csv(OUT_DIR / "dati" / "metadati_grokipedia.csv", index=False)

In [ ]:
# --- Internet Archive: v0.1 (ottobre 2025) e ultima cattura ------------------
def cdx(url, limite=3000):
    q = {"url": url, "output": "json", "fl": "timestamp", "limit": str(limite),
         "filter": "statuscode:200"}
    u = "http://web.archive.org/cdx/search/cdx?" + urllib.parse.urlencode(q)
    req = urllib.request.Request(u, headers={"User-Agent": UA_RICERCA})
    for k in range(4):
        try:
            with urllib.request.urlopen(req, timeout=WB_TIMEOUT, context=CTX) as r:
                b = r.read().decode("utf-8", errors="replace").strip()
            break
        except Exception:
            if k == 3: raise
            time.sleep(3 * (k + 1))
    if not b: return []
    d = json.loads(b)
    return [x[0] for x in (d[1:] if d and d[0][0] == "timestamp" else d)]

def snapshot(url, ts):
    """byte originali della cattura, decompressi se gzip"""
    fn = CACHE_WB / f"{hashlib.sha256((url+ts).encode()).hexdigest()[:16]}.bin"
    if fn.exists():
        return decodifica(fn.read_bytes())
    u = f"https://web.archive.org/web/{ts}id_/{url}"
    req = urllib.request.Request(u, headers={"User-Agent": UA_RICERCA})
    for k in range(3):
        try:
            with urllib.request.urlopen(req, timeout=WB_TIMEOUT, context=CTX) as r:
                b = r.read()
            break
        except Exception:
            if k == 2: raise
            time.sleep(3 * (k + 1))
    fn.write_bytes(b)
    time.sleep(0.8)
    return decodifica(b)

V01 = {}          # testo della prima cattura (v0.1) per ogni titolo
W = pd.DataFrame()
if FAI_WAYBACK:
    righe = []
    print("Internet Archive: prima cattura (v0.1) e ultima, per ogni articolo\\n")
    for i, t in enumerate(COPPIE, 1):
        url = "https://grokipedia.com/page/" + t.replace(" ", "_")
        r = dict(titolo=t, catture=np.nan, catture_2025=np.nan, prima=None, ultima=None,
                 v01_char=np.nan, ult_char=np.nan, v01_via=None,
                 sim_v01_attuale=np.nan, delta_char=np.nan, nota="")
        try:
            ts = cdx(url)
            r["catture"] = len(ts)
            r["catture_2025"] = sum(1 for x in ts if x < "20251201")
            if ts:
                r["prima"], r["ultima"] = ts[0], ts[-1]
                p = snapshot(url, ts[0])
                a, via = estrai_grok(p)
                r["v01_via"] = via
                if prosa_valida(a):
                    V01[t] = a
                    r["v01_char"] = len(a)
                    b = GROK[t]                     # versione attuale, gia' scaricata
                    r["ult_char"] = len(b)
                    sa, sb = shingles(a), shingles(b)
                    r["sim_v01_attuale"] = len(sa & sb) / len(sa | sb)
                    r["delta_char"] = len(b) - len(a)
                else:
                    r["nota"] = f"snapshot non valido ({len(a):,} char, via {via})"
            else:
                r["nota"] = "nessuna cattura"
        except Exception as e:
            r["nota"] = type(e).__name__
        righe.append(r)
        stato = (f"sim {r['sim_v01_attuale']:.3f}  {r['v01_char']:>8,.0f} -> {r['ult_char']:>8,.0f}"
                 if np.isfinite(r["sim_v01_attuale"]) else r["nota"])
        print(f"  {i:>2}/{len(COPPIE)} {t[:26]:26s} catture={r['catture']!s:>5} {stato}")
    W = pd.DataFrame(righe)
    W.to_csv(OUT_DIR / "dati" / "revisioni_wayback.csv", index=False)

    ok = W.dropna(subset=["sim_v01_attuale"])
    print(f"\\n  v0.1 recuperata per {len(ok)}/{len(W)} articoli")
    if len(ok):
        print(f"  catture per articolo: mediana {W['catture'].median():.0f} "
              f"[{W['catture'].min():.0f}-{W['catture'].max():.0f}]"
              f" | di cui nel 2025: mediana {W['catture_2025'].median():.0f}")
        print(f"  similarita' v0.1 <-> attuale ({SHINGLE_N}-grammi, Jaccard):")
        print(f"    mediana {ok['sim_v01_attuale'].median():.3f}  "
              f"[{ok['sim_v01_attuale'].min():.3f}, {ok['sim_v01_attuale'].max():.3f}]")
        print(f"  variazione di lunghezza: mediana {ok['delta_char'].median():+,.0f} caratteri"
              f"  [{ok['delta_char'].min():+,.0f}, {ok['delta_char'].max():+,.0f}]")
        print(f"\\n  quasi invariati (sim > 0.90): {int((ok['sim_v01_attuale']>0.9).sum())}/{len(ok)}")
        print(f"  riscritti a fondo (sim < 0.50): {int((ok['sim_v01_attuale']<0.5).sum())}/{len(ok)}")
        print("\\n  NB: e' una misura di quanto il TESTO e' cambiato fra la prima cattura")
        print("      archiviata e oggi, non un conteggio di edit: Grokipedia non lo espone.")

    # salva la v0.1 come terzo corpus
    if V01:
        dv = OUT_DIR / "corpus_v01"; dv.mkdir(exist_ok=True)
        for t, txt in V01.items():
            (dv / (re.sub(r"[^A-Za-z0-9_]", "", t.replace(" ", "_"))[:60] + ".txt")
             ).write_bytes(txt.encode("utf-8"))
        print(f"\\n  corpus v0.1 salvato in {dv} ({len(V01)} file)")

## 8. Riepilogo

In [ ]:
R = C.merge(S[["titolo", "grok_in_wiki"]], on="titolo", how="left")
if len(W):
    R = R.merge(W[["titolo", "catture", "catture_2025", "v01_char", "sim_v01_attuale", "delta_char"]],
                on="titolo", how="left")
R = R.merge(M[["titolo", "dateModified"]], on="titolo", how="left")
R["scritto_da_grok"] = R["grok_in_wiki"] <= SOGLIA_COPIA
R["usabile"] = R["scritto_da_grok"] & (R["wiki_tot"] >= N_EFF) & (R["grok_tot"] >= N_EFF)
R.to_csv(OUT_DIR / "dati" / "riepilogo.csv", index=False)

col = ["titolo", "wiki_tot", "grok_tot", "rapporto", "grok_in_wiki"]
if "catture" in R: col += ["catture", "v01_char", "sim_v01_attuale"]
print(R[col].to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

L = []
L.append("GROKIPEDIA vs WIKIPEDIA — 40 articoli appaiati per titolo")
L.append("=" * 68)
L.append(f"coppie complete: {len(R)} | finestra di analisi: {N_EFF:,} caratteri")
L.append(f"lunghezza mediana: wikipedia {R['wiki_tot'].median():,.0f} | "
         f"grokipedia {R['grok_tot'].median():,.0f} (rapporto {R['rapporto'].median():.2f})")
L.append("")
L.append(f"PATERNITA' (sovrapposizione di {SHINGLE_N}-grammi con Wikipedia)")
L.append(f"  mediana {R['grok_in_wiki'].median():.4f} | massimo {R['grok_in_wiki'].max():.4f}")
L.append(f"  scritti da Grok (<= {SOGLIA_COPIA:.0%}): {int(R['scritto_da_grok'].sum())}/{len(R)}")
L.append("")
L.append("REVISIONI")
L.append("  Grokipedia non pubblica una cronologia: nessun endpoint history/versions/revisions.")
if "catture" in R and R["catture"].notna().any():
    L.append(f"  catture su Internet Archive: mediana {R['catture'].median():.0f} per articolo")
    ok = R.dropna(subset=["sim_v01_attuale"])
    if len(ok):
        L.append(f"  v0.1 (ott 2025) recuperata per {len(ok)}/{len(R)} articoli")
        L.append(f"  similarita' v0.1 <-> attuale: mediana {ok['sim_v01_attuale'].median():.3f}")
        L.append(f"  invariati (>0.90): {int((ok['sim_v01_attuale']>0.9).sum())}/{len(ok)} | "
                 f"riscritti (<0.50): {int((ok['sim_v01_attuale']<0.5).sum())}/{len(ok)}")
        L.append(f"  variazione di lunghezza: mediana {ok['delta_char'].median():+,.0f} caratteri")
        L.append("  -> il corpus v0.1 e' salvato in risultati_confronto/corpus_v01 e puo'")
        L.append("     essere usato come TERZO corpus nell'analisi di Altmann.")
L.append("")
L.append(f"COPPIE UTILIZZABILI PER L'ANALISI: {int(R['usabile'].sum())}/{len(R)}")
testo = "\n".join(L)
(OUT_DIR / "riepilogo.txt").write_text(testo, encoding="utf-8")
print("\n" + testo)